## Installing Required Libraries

In [3]:
!pip install ipywidgets pdfplumber python-docx matplotlib

## Creating Cognitive Nexus — Master Pipeline

In [4]:
import os, json, time, requests, re
import numpy as np
import pdfplumber
from docx import Document
import ipywidgets as widgets
from IPython.display import display, Markdown

OLLAMA_MODEL = "qwen3.5:4b"

def call_llm(prompt, model=OLLAMA_MODEL, temperature=0.5, max_tokens=400, think=False):
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": model, "prompt": prompt, "stream": False, "think": think,
                  "options": {"temperature": temperature, "num_predict": max_tokens}},
            timeout=90
        )
        return response.json().get("response", "").strip()
    except Exception as e:
        return f"LLM call failed: {e}"

def get_embedding(text, model="nomic-embed-text:latest"):
    try:
        response = requests.post("http://localhost:11434/api/embeddings",
                                  json={"model": model, "prompt": text}, timeout=30)
        return response.json().get("embedding", [])
    except:
        return []

print("Cognitive Nexus — Master Pipeline Ready")

Cognitive Nexus — Master Pipeline Ready


## Creating Parsing Function

In [5]:
def extract_text_from_pdf(path):
    try:
        text = ""
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text.strip()
    except Exception:
        return ""

def extract_text_from_docx(path):
    try:
        doc = Document(path)
        return "\n".join([p.text for p in doc.paragraphs]).strip()
    except Exception:
        return ""

def extract_resume_text(path):
    if path.lower().endswith(".pdf"):
        return extract_text_from_pdf(path)
    elif path.lower().endswith(".docx"):
        return extract_text_from_docx(path)
    return ""

print("Parser functions loaded")

Parser functions loaded


## Creating Skill Extraction Function

In [6]:
TECHNICAL_SKILLS = [
    "Python", "Java", "JavaScript", "C++", "C", "SQL", "HTML", "CSS",
    "React", "Node.js", "FastAPI", "Django", "Flask", "MongoDB",
    "PostgreSQL", "MySQL", "AWS", "Azure", "Docker", "Kubernetes",
    "Git", "GitHub", "TypeScript", "Machine Learning", "Deep Learning",
    "AI", "Artificial Intelligence", "Data Science", "TensorFlow",
    "PyTorch", "NumPy", "Pandas", "LangChain", "REST API", "Linux"
]

def extract_skills_regex(text):
    text_lower = text.lower()
    return [s for s in TECHNICAL_SKILLS if s.lower() in text_lower]

def llm_extract_resume_data(resume_text):
    prompt = f"""Extract structured information from this resume. Return ONLY valid JSON, no other text.

Format:
{{
  "technical_skills": ["skill1", "skill2"],
  "soft_skills": ["skill1", "skill2"],
  "education": ["degree - institution - year"],
  "projects": [{{"name": "project name", "description": "one sentence summary"}}]
}}

RESUME:
{resume_text[:2000]}

Return ONLY the JSON object."""
    raw = call_llm(prompt, temperature=0.2, max_tokens=500)
    try:
        return json.loads(raw)
    except Exception:
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except Exception:
                return {}
        return {}

print("Skill extraction functions loaded")

Skill extraction functions loaded


## Creating Scoring Function

In [7]:
def calculate_ats_score(text, technical_skills):
    score = 0
    has_email = bool(re.search(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', text))
    has_phone = bool(re.search(r'\d{10}', text))
    if has_email: score += 10
    if has_phone: score += 10

    skill_count = len(technical_skills)
    if skill_count >= 8: score += 30
    elif skill_count >= 5: score += 20
    elif skill_count >= 1: score += 10

    section_keywords = ["experience", "education", "project", "skill", "certification", "achievement"]
    found = [kw for kw in section_keywords if kw in text.lower()]
    score += min(len(found) * 5, 30)

    word_count = len(text.split())
    if word_count >= 150: score += 20
    elif word_count >= 80: score += 10

    return {"ats_score": min(score, 100)}

def calculate_readiness_score(resume_score, dsa_score=0, projects_score=0,
                                communication_score=0, interview_score=0, consistency_score=0):
    weights = {"resume": 0.20, "dsa": 0.20, "projects": 0.20,
               "communication": 0.15, "interview": 0.15, "consistency": 0.10}
    total = (resume_score * weights["resume"] + dsa_score * weights["dsa"] +
             projects_score * weights["projects"] + communication_score * weights["communication"] +
             interview_score * weights["interview"] + consistency_score * weights["consistency"])
    breakdown = {"resume": round(resume_score, 2), "dsa": round(dsa_score, 2),
                 "projects": round(min(projects_score, 100), 2), "communication": round(communication_score, 2),
                 "interview": round(interview_score, 2), "consistency": round(consistency_score, 2)}
    weak_areas = [k for k, v in breakdown.items() if v < 50]
    return {"career_readiness_score": round(total, 2), "breakdown": breakdown, "weak_areas": weak_areas}

print("Scoring functions loaded")

Scoring functions loaded


## Creating Gap, roadmap, and certification functions

In [8]:
JOB_ROLE_SKILLS = {
    "AI Engineer": ["python", "machine learning", "deep learning", "tensorflow", "pytorch",
                    "langchain", "docker", "kubernetes", "fastapi", "sql", "rest api", "nlp", "rag"],
    "Backend Developer": ["python", "java", "sql", "fastapi", "django", "flask", "docker", "git"],
    "Full Stack Developer": ["javascript", "html", "css", "react", "node.js", "sql", "git"],
    "Data Scientist": ["python", "sql", "pandas", "numpy", "machine learning", "tensorflow", "statistics"]
}

def analyze_skill_gap(your_skills, target_role):
    required = set(s.lower() for s in JOB_ROLE_SKILLS.get(target_role, []))
    yours = set(s.lower() for s in your_skills)
    matched = yours & required
    missing = required - yours
    match_pct = round((len(matched) / len(required)) * 100, 1) if required else 0
    return {"target_role": target_role, "matched_skills": sorted(matched),
            "missing_skills": sorted(missing), "match_percentage": match_pct}

SKILL_RESOURCES = {
    "docker": {"time_weeks": 1, "resource": "Docker Mastery course"},
    "kubernetes": {"time_weeks": 2, "resource": "Kubernetes basics on KodeKloud"},
    "fastapi": {"time_weeks": 1, "resource": "FastAPI official tutorial"},
    "langchain": {"time_weeks": 2, "resource": "LangChain docs + build a RAG project"},
    "rag": {"time_weeks": 2, "resource": "Build a RAG pipeline with ChromaDB"},
    "nlp": {"time_weeks": 2, "resource": "NLP with Python — Hugging Face"},
    "deep learning": {"time_weeks": 3, "resource": "DeepLearning.AI Specialization"},
    "rest api": {"time_weeks": 1, "resource": "Build REST APIs with FastAPI"},
}
DEFAULT_RESOURCE = {"time_weeks": 1, "resource": "Documentation + build a small project"}

def generate_roadmap(missing_skills):
    roadmap, week = [], 1
    for skill in missing_skills:
        info = SKILL_RESOURCES.get(skill.lower(), DEFAULT_RESOURCE)
        weeks = info["time_weeks"]
        duration = f"Week {week}" if weeks == 1 else f"Week {week}-{week + weeks - 1}"
        roadmap.append({"skill": skill, "duration": duration, "resource": info["resource"]})
        week += weeks
    return {"roadmap": roadmap, "total_duration_weeks": week - 1}

CERTIFICATION_DATABASE = {
    "docker": [{"name": "Docker Certified Associate", "provider": "Docker Inc."}],
    "kubernetes": [{"name": "CKAD", "provider": "CNCF"}],
    "fastapi": [{"name": "FastAPI Course Certificate", "provider": "Udemy"}],
    "deep learning": [{"name": "Deep Learning Specialization", "provider": "DeepLearning.AI"}],
    "langchain": [{"name": "LangChain for LLM App Development", "provider": "DeepLearning.AI"}],
}

def recommend_certifications(missing_skills):
    recommended, seen = [], set()
    for skill in missing_skills:
        for cert in CERTIFICATION_DATABASE.get(skill, []):
            if cert["name"] not in seen:
                recommended.append({**cert, "for_skill": skill})
                seen.add(cert["name"])
    return recommended

print("Gap, roadmap, and certification functions loaded")

Gap, roadmap, and certification functions loaded


## Loading Resume File from Sample Resumes

In [9]:
uploaded_filename = "Resume.pdf"
uploaded_path = "../sample_resume/Resume.pdf"

if os.path.exists(uploaded_path):
    print(f"✅ Using resume: {uploaded_filename}")
    print(f"Path: {uploaded_path}")
else:
    print(f"❌ File not found at {uploaded_path} — check sample_resume folder")

✅ Using resume: Resume.pdf
Path: ../sample_resume/Resume.pdf


## Running Cognitive Nexus Pipeline

In [10]:
def run_full_pipeline(resume_path, target_role="AI Engineer"):
    print("🚀 Running Cognitive Nexus pipeline...\n")

    print("[1/8] Parsing resume...")
    resume_text = extract_resume_text(resume_path)

    print("[2/8] Extracting skills (regex + LLM)...")
    regex_skills = extract_skills_regex(resume_text)
    llm_data = llm_extract_resume_data(resume_text)
    technical_skills = llm_data.get("technical_skills", regex_skills) or regex_skills

    print("[3/8] Calculating ATS + readiness score...")
    ats_result = calculate_ats_score(resume_text, technical_skills)
    projects_score = min(len(llm_data.get("projects", [])) * 20, 100)
    readiness_result = calculate_readiness_score(resume_score=ats_result["ats_score"], projects_score=projects_score)

    print("[4/8] Analyzing skill gap...")
    gap_result = analyze_skill_gap(technical_skills, target_role)

    print("[5/8] Generating learning roadmap...")
    roadmap_result = generate_roadmap(gap_result["missing_skills"])

    print("[6/8] Recommending certifications...")
    certs_result = recommend_certifications(gap_result["missing_skills"])

    print("[7/8] Generating interview questions (LLM)...")
    interview_prompt = f"""Generate 5 interview questions for a {target_role} role based on this resume. Numbered list only.

RESUME: {resume_text[:1500]}"""
    interview_questions = call_llm(interview_prompt, temperature=0.6, max_tokens=350)

    print("[8/8] Building RAG knowledge base...")
    chunks = [
        {"source": "resume", "text": f"Resume: {resume_text[:1000]}"},
        {"source": "skills", "text": f"Technical skills: {technical_skills}. Soft skills: {llm_data.get('soft_skills', [])}"},
        {"source": "readiness", "text": f"Career readiness: {json.dumps(readiness_result)}"},
        {"source": "ats", "text": f"ATS analysis: {json.dumps(ats_result)}"},
        {"source": "skill_gap", "text": f"Skill gap for {target_role}: {json.dumps(gap_result)}"},
        {"source": "roadmap", "text": f"Learning roadmap: {json.dumps(roadmap_result)}"},
        {"source": "certifications", "text": f"Certifications: {json.dumps(certs_result)}"},
        {"source": "interview", "text": f"Interview questions: {interview_questions}"},
        {"source": "projects", "text": f"Projects: {json.dumps(llm_data.get('projects', []))}"},
    ]
    for chunk in chunks:
        chunk["embedding"] = get_embedding(chunk["text"])

    print("\n✅ Pipeline complete!\n")
    return {
        "resume_text": resume_text, "technical_skills": technical_skills,
        "soft_skills": llm_data.get("soft_skills", []), "ats": ats_result,
        "readiness": readiness_result, "skill_gap": gap_result, "roadmap": roadmap_result,
        "certifications": certs_result, "interview_questions": interview_questions,
        "projects": llm_data.get("projects", []), "rag_chunks": chunks
    }

start = time.time()
pipeline_result = run_full_pipeline(uploaded_path, target_role="AI Engineer")
print(f"⏱️ Total pipeline time: {time.time() - start:.1f} seconds")

🚀 Running Cognitive Nexus pipeline...

[1/8] Parsing resume...
[2/8] Extracting skills (regex + LLM)...
[3/8] Calculating ATS + readiness score...
[4/8] Analyzing skill gap...
[5/8] Generating learning roadmap...
[6/8] Recommending certifications...
[7/8] Generating interview questions (LLM)...
[8/8] Building RAG knowledge base...

✅ Pipeline complete!

⏱️ Total pipeline time: 12.9 seconds


## Collective Output of all Pipelines

In [11]:
print("="*60)
print(f"CAREERFORGE AI — RESULTS FOR {uploaded_filename}")
print("="*60)
print(f"\n📊 Career Readiness Score: {pipeline_result['readiness']['career_readiness_score']}/100")
print(f"📄 ATS Score: {pipeline_result['ats']['ats_score']}/100")
print(f"🎯 Skill Match ({pipeline_result['skill_gap']['target_role']}): {pipeline_result['skill_gap']['match_percentage']}%")
print(f"🛠️ Technical Skills Found: {len(pipeline_result['technical_skills'])}")
print(f"📚 Learning Roadmap: {pipeline_result['roadmap']['total_duration_weeks']} weeks")
print(f"\nMissing Skills: {', '.join(pipeline_result['skill_gap']['missing_skills'])}")

CAREERFORGE AI — RESULTS FOR Resume.pdf

📊 Career Readiness Score: 27.0/100
📄 ATS Score: 95/100
🎯 Skill Match (AI Engineer): 30.8%
🛠️ Technical Skills Found: 13
📚 Learning Roadmap: 15 weeks

Missing Skills: deep learning, docker, fastapi, kubernetes, langchain, machine learning, nlp, rag, rest api


## Chat Interface with RAG Integration

In [14]:
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0:
        return 0
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def rag_chat(query, chunks, top_k=3):
    query_embedding = get_embedding(query)
    scored = [(cosine_similarity(query_embedding, c["embedding"]), c) for c in chunks if c["embedding"]]
    scored.sort(key=lambda x: x[0], reverse=True)
    top_chunks = [c for _, c in scored[:top_k]]
    context = "\n\n".join([f"[{c['source']}]: {c['text']}" for c in top_chunks])

    prompt = f"""You are Cognitive Nexus, a warm, specific career mentor. Answer using ONLY this context, under 130 words.

CONTEXT:
{context}

QUESTION: {query}

Answer:"""
    return call_llm(prompt, temperature=0.6, max_tokens=300)

chat_input = widgets.Text(placeholder='Ask Cognitive Nexus anything...', layout=widgets.Layout(width='80%'))
chat_button = widgets.Button(description='Send', button_style='success')
chat_output = widgets.Output()

def on_send(b):
    with chat_output:
        query = chat_input.value
        if query.strip():
            print(f"You: {query}")
            answer = rag_chat(query, pipeline_result["rag_chunks"])
            print(f"Cognitive Nexus: {answer}\n")
            chat_input.value = ""

chat_button.on_click(on_send)
display(Markdown("## 💬 Chat with Cognitive Nexus"))
display(widgets.HBox([chat_input, chat_button]), chat_output)

## 💬 Chat with Cognitive Nexus

Output()